In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import heapq as hq
from time import time
import random

In [2]:
TM_DESC = (2, 8, 4)

In [109]:
def mem_str(memory):
    result = "MEM: "
    for i in range(memory.shape[0]):
        result += str(int(memory[i])) + " "
    return result

In [3]:
def getHeuristicModel(width=128, layers=16):
    #Input: (write_symbol, next_state, move, read_symbol, current_state)
    model = nn.Sequential(nn.Linear(5, width))
    model.append(nn.ReLU())

    for _ in range(layers):
        model.append(nn.Linear(width, width))
        model.append(nn.ReLU())
    
    model.append(nn.Linear(width, 1))

    return model

In [4]:
heuristic = getHeuristicModel()
trans = torch.tensor([0, 4, -3, 1, 2], dtype=torch.float)
print(heuristic(trans))

tensor([-0.0989], grad_fn=<ViewBackward0>)


In [ ]:
def transition(memory: torch.Tensor, state: torch.Tensor, action: torch.Tensor):
    # Memory: [...] - The TM's read/write memory tape
    # State: (position, current_state) - The TM's head position on the tape and it's internal state
    # Action: (write_symbol, next_state, move, read_symbol, current_state) - The action taken by the TM. The symbol to write, the next state, how far to move the head

    new_state = state.clone().detach()
    new_state[1] = action[1]
    new_state[0] += action[2]

    mem_size = memory.shape[0]
    padding_right = 0
    while new_state[0] >= mem_size:
        padding_right += mem_size
        mem_size += mem_size

    new_memory = nn.functional.pad(memory, (0, padding_right))
    new_memory[int(new_state[0] - action[2])] = action[0]
    
    if new_state[0] < 0:
        new_state[0] = 0

    return (new_memory, new_state)    

In [6]:
mem = torch.tensor(
    [0, 1, 2, 3, 4, 5]
)

state = torch.tensor(
    [1, 5],
)

action = torch.tensor(
    [9, 3, 20],
)

new_mem, new_state = transition(mem, state, action)

print(new_mem)
print(new_state)

tensor([0, 9, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
tensor([21,  3])


In [7]:
def enumerate_actions(read_symbol, current_state):
    for symbol in range(TM_DESC[0]):
        for state in range(TM_DESC[1]):
            for move in range(TM_DESC[2]):
                for sign in [-1, 1]:
                    yield torch.tensor([symbol, state, (move + 1)*sign, read_symbol, current_state], dtype=torch.float)

class TMScoredTransition:
    def __init__(self, heuristic, last_transition, memory: torch.Tensor, state: torch.Tensor, action: torch.Tensor):
        self.score = heuristic(action)
        self.last_transition = last_transition
        if(last_transition == None):
            self.depth = 0
        else:
            self.depth = last_transition.depth + 1
        self.memory = memory
        self.state = state
        self.action = action
    
    def __lt__(self, other):
        return self.score < other.score

def enumerate_transitions(heuristic, last_transition: TMScoredTransition, memory, state):
    read_symbol = memory[int(state[0])]
    current_state = state[1]
    actions = enumerate_actions(read_symbol, current_state)
    for action in actions:
        yield TMScoredTransition(heuristic, last_transition, memory, state, action)


In [8]:
def tm_search(heuristic, initial_transitions, expansions=128):
    #TM_desc: (symbols, states, move_range) - the number of symbols, number of states, and possible move_range of the TM
    frontier = initial_transitions
    hq.heapify(frontier)

    last_transition = None
    for i in range(expansions):
        last_transition = hq.heappop(frontier)
        new_memory, new_state = transition(last_transition.memory, last_transition.state, last_transition.action)

        for trans in enumerate_transitions(heuristic, last_transition, new_memory, new_state):
            hq.heappush(frontier, trans)
    
    return last_transition

In [9]:
heuristic = getHeuristicModel()
input_memory = torch.tensor([0, 0, 0, 0], dtype=torch.float)
initial_state = torch.tensor([0, 0], dtype=torch.float)
initial_transitions = [trans for trans in enumerate_transitions(heuristic, None, input_memory, initial_state)]

last_transition = tm_search(heuristic, initial_transitions)
print(last_transition.score)
print(last_transition.depth)
print(last_transition.memory)

tensor([-0.0258], grad_fn=<ViewBackward0>)
127
tensor([0., 0., 0., 0.])


In [ ]:
def get_distance_heuristic(initial_state, input_memory, target_memory):
    distance_function = nn.L1Loss(reduction = 'sum')
    last_memory = input_memory
    last_state = initial_state

    def heuristic(action):
        new_memory, new_state = transition(last_memory, last_state, action)
        last_memory = new_memory
        last_state = new_state

        memory_size = new_memory.shape[0]
        target_size = target_memory.shape[0]
        if memory_size < target_size:
                new_memory = nn.functional.pad(new_memory, (0, target_size - memory_size))
        elif memory_size > target_size:
            target_memory = nn.functional.pad(target_memory, (0, memory_size - target_size))

        return distance_function(new_memory, target_memory)

    return heuristic

In [ ]:
def get_distance_training_data(task_generator, batch_count = 64, batch_size = 128):
    # TM_desc: (symbols, states, move_range)
    # Action: (write_symbol, next_state, move, read_symbol, current_state)
    distance_function = nn.L1Loss(reduction = 'sum')

    for _ in range(batch_count):
        actions_batch = torch.zeros(batch_size, 5)
        distances_batch = torch.zeros(batch_size, 1)
        
        input_memory, target_memory = task_generator()

        last_state = torch.tensor([0, 0], dtype=torch.float)
        last_memory = input_memory

        for i in range(batch_size):
            read_symbol = last_memory[int(last_state[0])]
            current_state = last_state[1]
            action = torch.tensor([
                random.randint(0, TM_DESC[0] - 1), 
                random.randint(0, TM_DESC[1] - 1), 
                random.randint(1, TM_DESC[2])*random.choice([-1, 1]), 
                read_symbol, current_state], dtype=torch.float)
            new_memory, new_state = transition(last_memory, last_state, action)
            last_state = new_state
            last_memory = new_memory

            memory_size = new_memory.shape[0]
            target_size = target_memory.shape[0]
            if memory_size < target_size:
                new_memory = nn.functional.pad(new_memory, (0, target_size - memory_size))
            elif memory_size > target_size:
                target_memory = nn.functional.pad(target_memory, (0, memory_size - target_size))

            distance = distance_function(new_memory, target_memory)
            actions_batch[i] = action
            distances_batch[i] = distance
            print(mem_str(last_memory), "DIST: ", float(distance))

        yield (actions_batch, distances_batch)



In [101]:
def copy_task_generator(max_size=6):
    size = random.randint(1, max_size)
    input_vector = torch.zeros([1], dtype=torch.float)
    target_vector = torch.full([size*2 + 1], 1, dtype=torch.float)
    target_vector[size] = 0

    return (input_vector, target_vector)

In [115]:
for action, dist in get_distance_training_data(copy_task_generator):
    pass

MEM: 0  DIST:  2.0
MEM: 0 0  DIST:  2.0
MEM: 0 1  DIST:  3.0
MEM: 1 1  DIST:  2.0
MEM: 0 1  DIST:  3.0
MEM: 0 1  DIST:  3.0
MEM: 0 1  DIST:  3.0
MEM: 0 1 0 0 0 0 0 0  DIST:  3.0
MEM: 0 1 0 0 0 0 0 0  DIST:  3.0
MEM: 1 1 0 0 0 0 0 0  DIST:  2.0
MEM: 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0  DIST:  3.0
MEM: 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0  DIST:  4.0
MEM: 1 1 0 0 1 0 0 0 1 0 0 0 1 0 0 0  DIST:  5.0
MEM: 1 1 0 0 1 0 0 0 1 0 1 0 1 0 0 0  DIST:  6.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 1 0 0 0  DIST:  7.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 0 0 0  DIST:  6.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0  DIST:  7.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0  DIST:  8.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0  DIST:  8.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0  DIST:  8.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0  DIST:  9.0
MEM: 1 1 0 0 1 0 0 0 1 1 1 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 

In [ ]:
def train_heuristic(heuristic, training_data, epochs = 64):
    loss_function = nn.MSELoss()
    optimizer = optim.SGD(heuristic.parameters())
    start_time = time()

    for i in range(epochs):
        running_loss = 0
        training_samples = 0

        for transitions, values in training_data:
            output = heuristic(transitions)
            loss = loss_function(output, values)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            training_samples += 1
        
        print(f"Epoch {i}/{epochs} - Loss {running_loss/training_samples} - Time {time() - start_time}s")